# 04 · Eleição do vencedor e promoção controlada

**Etapas do fluxo:** 7 (avaliação comparativa e eleição), 8 (registro no MLflow) e 9 (regra de promoção).

1. O vencedor do ciclo é o candidato com maior ROC-AUC na validação (empate → Modelo A, mais simples).
2. O **modelo vigente** (carregado do repositório, não do runner) é reavaliado **na mesma validação do ciclo** -
   mesmo protocolo, mesmos registros.
3. **Regra de promoção (config.yaml):** promove se `AUC_vencedor >= AUC_vigente + melhoria_minima` **e**
   `AUC_vencedor >= metrica_minima` **e** as verificações de qualidade passaram. Caso contrário, o vigente é mantido.
4. Somente aqui o estado do ciclo é efetivado (lotes incorporados, manifesto, histórico): se qualquer notebook anterior
   falhar, nada disso é alterado.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [2]:
import mlflow
from src import ingestao
from src.ciclo import atualizar_contexto, carregar_contexto, diretorio_ciclo, obter_cycle_id, salvar_json
from src.modelagem import avaliar, colunas_entrada
from src.promocao import (
    PROMOVER, atualizar_manifesto, carregar_manifesto, carregar_modelo_vigente, decidir_promocao,
    eleger_vencedor, registrar_historico, salvar_modelo_promovido,
)
from src.rastreamento import configurar_mlflow, marcar_ciclo, run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
processado = caminho(cfg, cfg["dados"]["diretorio_processado"])
validacao = pd.read_csv(processado / "validacao.csv")
teste = pd.read_csv(processado / "teste.csv")
metrica, direcao = cfg["avaliacao"]["metrica_principal"], cfg["avaliacao"]["direcao"]
alvo, X_cols = cfg["dados"]["alvo"], colunas_entrada(cfg)

## Verificações antes de decidir

In [3]:
configurar_mlflow(cfg)
candidatos = contexto.get("candidatos", {})
esperados = set(cfg["modelos"])
assert set(candidatos) == esperados, f"Candidatos ausentes: {esperados - set(candidatos)}"
for chave, dados in candidatos.items():
    run = mlflow.get_run(dados["run_id"])
    assert run.info.status == "FINISHED", f"Run de {chave} não terminou: {run.info.status}"
    assert run.data.tags.get("cycle_id") == cycle_id, f"Run de {chave} pertence a outro ciclo"
print("Os dois candidatos foram treinados neste ciclo:", {c: d["run_id"] for c, d in candidatos.items()})

Os dois candidatos foram treinados neste ciclo: {'modelo_a': '9a991687171146b0a36ff6d5962cf497', 'modelo_b': '07bbc079709d4c02a58c488cfbc7fdfe'}


In [4]:
metricas = {chave: dados["metricas_validacao"][metrica] for chave, dados in candidatos.items()}
vencedor = eleger_vencedor(metricas, direcao)
modelo_vencedor = mlflow.sklearn.load_model(f"runs:/{candidatos[vencedor]['run_id']}/modelo")
# Reavaliar o modelo recarregado confirma que o artefato registrado é o mesmo que foi avaliado.
metricas_vencedor = avaliar(modelo_vencedor, validacao[X_cols], validacao[alvo])
assert abs(metricas_vencedor[metrica] - metricas[vencedor]) < 1e-9

vigente, manifesto = carregar_modelo_vigente(cfg)
metricas_vigente = avaliar(vigente, validacao[X_cols], validacao[alvo]) if vigente is not None else None
print("Métricas dos candidatos:", metricas, "-> vencedor do ciclo:", vencedor, candidatos[vencedor]["nome"])
print("Vigente atual:", manifesto["cycle_id"] + " / " + manifesto["algoritmo"] if manifesto else "nenhum")
print(f"{metrica} do vigente nesta validação:", metricas_vigente[metrica] if metricas_vigente else "-")

Métricas dos candidatos: {'modelo_a': 0.799, 'modelo_b': 0.8064} -> vencedor do ciclo: modelo_b hist_gradient_boosting
Vigente atual: c20261123T183614Z-gh45240022843 / hist_gradient_boosting
roc_auc do vigente nesta validação: 0.7971


In [5]:
decisao = decidir_promocao(
    metrica_candidato=metricas_vencedor[metrica],
    metrica_vigente=metricas_vigente[metrica] if metricas_vigente else None,
    regra=cfg["promocao"],
    direcao=direcao,
    qualidade_ok=bool(contexto.get("qualidade_ok")),
)
print(json.dumps(decisao, indent=2, ensure_ascii=False))

novo_vigente = None
if decisao["decisao"] == PROMOVER:
    caminho_modelo = salvar_modelo_promovido(modelo_vencedor, cfg, cycle_id, candidatos[vencedor]["nome"])
    novo_vigente = {
        "cycle_id": cycle_id,
        "data_version": contexto["data_version"],
        "candidato": vencedor,
        "algoritmo": candidatos[vencedor]["nome"],
        "run_id": candidatos[vencedor]["run_id"],
        "experimento": cfg["mlflow"]["experimento"],
        "features": contexto["features_eleitas"],
        "metrica": {"nome": metrica, "validacao": metricas_vencedor[metrica]},
        "caminho_modelo": caminho_modelo,
    }
manifesto = atualizar_manifesto(cfg, manifesto, contexto, decisao, novo_vigente)

{
  "decisao": "manter_vigente",
  "motivo": "Ganho de +0.0093 < limiar 0.01.",
  "criterios": {
    "qualidade_ok": true,
    "acima_do_piso": true,
    "metrica_candidato": 0.8064,
    "metrica_vigente": 0.7971,
    "piso": 0.6,
    "melhoria_minima": 0.01,
    "ganho": 0.0093,
    "supera_limiar": false
  }
}


## Avaliação final no conjunto de teste reservado
Apenas **relatada** - não participa de nenhuma decisão. Mede o modelo que ficou vigente após o ciclo.

In [6]:
vigente_final, manifesto = carregar_modelo_vigente(cfg)
metricas_teste = avaliar(vigente_final, teste[X_cols], teste[alvo]) if vigente_final is not None else {}
if metricas_teste:
    manifesto["ultimo_ciclo"]["teste_vigente"] = metricas_teste
    caminho(cfg, cfg["promocao"]["manifesto"]).write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding="utf-8")
print("Teste do vigente:", metricas_teste)

Teste do vigente: {'f1_macro': 0.6674, 'balanced_accuracy': 0.7574, 'accuracy': 0.8099, 'precision_pos': 0.3337, 'recall_pos': 0.6897, 'n': 4119, 'taxa_positiva_real': 0.1126, 'roc_auc': 0.811}


## Efetivação do ciclo: estado de ingestão, histórico, MLflow e resumo

In [7]:
estado = ingestao.carregar_estado(cfg)
estado["lotes_incorporados"] = contexto["lotes_pool"]
estado["versoes"].append({"cycle_id": cycle_id, "data_version": contexto["data_version"], "lotes": contexto["lotes_pool"]})
ingestao.salvar_estado(cfg, estado)

registro = {
    "cycle_id": cycle_id,
    "data_version": contexto["data_version"],
    "lotes_novos": "|".join(contexto["lotes_novos"]) or "nenhum",
    "n_treino": contexto["n_treino"],
    "n_validacao": contexto["n_validacao"],
    "features_eleitas": "|".join(contexto["features_eleitas"]),
    "n_features": len(contexto["features_eleitas"]),
    **{f"{metrica}_{c}": round(v, 4) for c, v in metricas.items()},
    "vencedor": candidatos[vencedor]["nome"],
    f"{metrica}_vigente_anterior": metricas_vigente[metrica] if metricas_vigente else "",
    "decisao": decisao["decisao"],
    "vigente_apos": f"{manifesto['cycle_id']}__{manifesto['algoritmo']}" if manifesto.get("cycle_id") else "",
    f"{metrica}_teste_vigente": metricas_teste.get(metrica, ""),
    "drift_alerta": contexto.get("drift_alerta", False),
}
registrar_historico(cfg, registro)
salvar_json(pasta_ciclo / "decisao.json", {"vencedor": vencedor, "metricas_candidatos": metricas, "metricas_vigente_anterior": metricas_vigente, **decisao, "teste_vigente": metricas_teste})

with run_etapa(cfg, contexto, "decisao_promocao"):
    mlflow.log_params({"vencedor": candidatos[vencedor]["nome"], "decisao": decisao["decisao"], "motivo": decisao["motivo"][:250], "regra_melhoria_minima": cfg["promocao"]["melhoria_minima"], "regra_metrica_minima": cfg["promocao"]["metrica_minima"]})
    mlflow.log_metrics({f"val_{metrica}_vencedor": metricas_vencedor[metrica], "promovido": int(decisao["decisao"] == PROMOVER)})
    if metricas_vigente:
        mlflow.log_metric(f"val_{metrica}_vigente_anterior", metricas_vigente[metrica])
    if metricas_teste:
        mlflow.log_metric(f"teste_{metrica}_vigente", metricas_teste[metrica])
    mlflow.log_artifact(str(pasta_ciclo / "decisao.json"))
    mlflow.log_artifact(str(caminho(cfg, cfg["promocao"]["manifesto"])))
marcar_ciclo(cfg, contexto, {"decisao": decisao["decisao"], "vencedor": candidatos[vencedor]["nome"], "vigente_run_id": manifesto.get("run_id", "")})
contexto = atualizar_contexto(cfg, cycle_id, vencedor=vencedor, decisao=decisao["decisao"])

In [8]:
drift = json.loads((pasta_ciclo / "drift.json").read_text(encoding="utf-8"))
linhas = [
    f"# Ciclo `{cycle_id}`",
    "",
    f"- **data_version:** `{contexto['data_version']}` (lotes novos: {', '.join(contexto['lotes_novos']) or 'nenhum'})",
    f"- **Features eleitas ({len(contexto['features_eleitas'])}):** {', '.join(contexto['features_eleitas'])}",
    "",
    f"| Candidato | {metrica} validação | run_id |",
    "|---|---|---|",
    *[f"| {c} · {d['nome']} | {d['metricas_validacao'][metrica]:.4f} | `{d['run_id']}` |" for c, d in candidatos.items()],
    "",
    f"- **Vencedor do ciclo:** {candidatos[vencedor]['nome']}",
    f"- **Vigente anterior nesta validação:** {metricas_vigente[metrica]:.4f}" if metricas_vigente else "- **Vigente anterior:** nenhum",
    f"- **Decisão:** `{decisao['decisao']}` - {decisao['motivo']}",
    f"- **Vigente após o ciclo:** {manifesto.get('algoritmo')} (ciclo `{manifesto.get('cycle_id')}`, run `{manifesto.get('run_id')}`)",
    f"- **{metrica} do vigente no teste reservado:** {metricas_teste.get(metrica, '-')}",
    f"- **Drift:** {'ALERTA - ' + ', '.join(drift.get('features_em_alerta', [])) if drift.get('alerta') else 'sem alerta'}",
]
(pasta_ciclo / "resumo.md").write_text("\n".join(linhas) + "\n", encoding="utf-8")
print("\n".join(linhas))

# Ciclo `c20261123T185659Z-gh45243588298`

- **data_version:** `v005-6563af51` (lotes novos: nenhum)
- **Features eleitas (9):** euribor3m, contatado_antes, emp.var.rate, cons.price.idx, poutcome, contact, nr.employed, month, cons.conf.idx

| Candidato | roc_auc validação | run_id |
|---|---|---|
| modelo_a · logistic_regression | 0.7990 | `9a991687171146b0a36ff6d5962cf497` |
| modelo_b · hist_gradient_boosting | 0.8064 | `07bbc079709d4c02a58c488cfbc7fdfe` |

- **Vencedor do ciclo:** hist_gradient_boosting
- **Vigente anterior nesta validação:** 0.7971
- **Decisão:** `manter_vigente` - Ganho de +0.0093 < limiar 0.01.
- **Vigente após o ciclo:** hist_gradient_boosting (ciclo `c20261123T183614Z-gh45240022843`, run `b5dc8f68a52a44d4952e8171783d6c09`)
- **roc_auc do vigente no teste reservado:** 0.811
- **Drift:** sem alerta
